# 面试问题：代码库 RAG 怎样按符号与依赖检索上下文，并避免把过期代码喂给 Code Agent？

**一句话回答。** 以 function/class/module 等语义符号切块，保存文件路径、行范围、导入/调用边、commit revision 和测试关联；先检索直接符号，再在有限预算内展开依赖。回答或 patch 必须引用当前 revision 的代码片段，不能用相似文本替代真实 API 契约。

本 Notebook 以小型、受控数据实现必要的数据合同、验证器和状态机。它不访问真实网站、文件或模型，也不把断言结果宣传成生产质量、安全保证或法律合规结论。

**资料入口。** [RepoCoder](https://arxiv.org/abs/2303.12570) 使用迭代检索辅助仓库级代码生成；本例实现 symbol/dependency/provenance 的最小控制面。


In [ ]:
question = "Code RAG 符号检索"  # 执行本行的状态、计算或校验逻辑。
assert "RAG" in question  # 执行本行的状态、计算或校验逻辑。
assert 11 % 2 == 1  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 代码 chunk 按符号而非固定字符窗口组织

函数签名、类、导入和测试共同构成 API 语义。固定长度切块常把定义与调用拆开；symbol chunk 至少要有 path、symbol、行范围、语言、revision 和文本，重命名/移动时也能追踪。


In [ ]:
symbols = [{"id": "s1", "path": "refund.py", "name": "apply_refund", "range": (10, 18), "text": "def apply_refund(order): return check_limit(order)", "revision": "c1"}, {"id": "s2", "path": "limits.py", "name": "check_limit", "range": (3, 8), "text": "def check_limit(order): return order.amount < 100", "revision": "c1"}, {"id": "s3", "path": "tests/test_refund.py", "name": "test_limit", "range": (1, 6), "text": "def test_limit(): assert check_limit(Order(20))", "revision": "c1"}]  # 执行本行的状态、计算或校验逻辑。
assert len(symbols) == 3  # 执行本行的状态、计算或校验逻辑。
assert all(item["range"][0] < item["range"][1] for item in symbols)  # 执行本行的状态、计算或校验逻辑。
assert {item["revision"] for item in symbols} == {"c1"}  # 执行本行的状态、计算或校验逻辑。

## 2. 依赖图补足定义—调用之间的关系

检索到 apply_refund 后，Agent 通常还需要 check_limit 和相应测试。依赖边应来自解析器/编译器或受控静态分析，并标注边类型与 revision；LLM 猜出的 import 不能直接当作事实。


In [ ]:
edges = [{"from": "s1", "to": "s2", "kind": "call", "revision": "c1"}, {"from": "s3", "to": "s2", "kind": "call", "revision": "c1"}]  # 执行本行的状态、计算或校验逻辑。
assert len(edges) == 2  # 执行本行的状态、计算或校验逻辑。
assert all(edge["revision"] == "c1" for edge in edges)  # 执行本行的状态、计算或校验逻辑。
assert edges[0]["kind"] == "call"  # 执行本行的状态、计算或校验逻辑。

## 3. 直接符号检索先于依赖扩展

教学用 query/token 字符重叠模拟 code retriever。真实系统可融合标识符、注释、BM25、code embedding 和调用图，但先找直接 match 再扩展能减少无关依赖占满上下文窗口。


In [ ]:
def score(query, symbol):  # 执行本行的状态、计算或校验逻辑。
    return len(set(query.lower()) & set((symbol["name"] + " " + symbol["text"]).lower()))  # 执行本行的状态、计算或校验逻辑。
def retrieve(query):  # 执行本行的状态、计算或校验逻辑。
    return sorted(symbols, key=lambda symbol: score(query, symbol), reverse=True)  # 执行本行的状态、计算或校验逻辑。
direct = retrieve("apply_refund")  # 执行本行的状态、计算或校验逻辑。
assert direct[0]["id"] == "s1"  # 执行本行的状态、计算或校验逻辑。
assert score("apply_refund", direct[0]) > score("apply_refund", direct[1])  # 执行本行的状态、计算或校验逻辑。
assert len(direct) == 3  # 执行本行的状态、计算或校验逻辑。

## 4. 依赖扩展必须有 hop 和 token 预算

盲目递归调用图会把公共工具、整个框架甚至循环依赖塞进 prompt。扩展器应记录 parent、hop 和原因，并优先保留直接符号；遇到超过预算时向模型报告缺失依赖，而不是假装上下文完整。


In [ ]:
symbol_by_id = {symbol["id"]: symbol for symbol in symbols}  # 执行本行的状态、计算或校验逻辑。
def expand(seed_ids, max_hops):  # 执行本行的状态、计算或校验逻辑。
    selected = list(seed_ids)  # 执行本行的状态、计算或校验逻辑。
    frontier = list(seed_ids)  # 执行本行的状态、计算或校验逻辑。
    for hop in range(max_hops):  # 执行本行的状态、计算或校验逻辑。
        frontier = [edge["to"] for edge in edges if edge["from"] in frontier and edge["to"] not in selected]  # 执行本行的状态、计算或校验逻辑。
        selected.extend(frontier)  # 执行本行的状态、计算或校验逻辑。
    return selected  # 执行本行的状态、计算或校验逻辑。
expanded_ids = expand(["s1"], 1)  # 执行本行的状态、计算或校验逻辑。
assert expanded_ids == ["s1", "s2"]  # 执行本行的状态、计算或校验逻辑。
assert expand(["s1"], 0) == ["s1"]  # 执行本行的状态、计算或校验逻辑。

## 5. 测试和实现应作为不同类型的上下文

代码 Agent 修改实现时，相关测试既是约束也是验证入口。测试不应被误当成生产 API 定义；context assembly 应标注角色，避免模型把 fixture、mock 或过期断言当业务规则。


In [ ]:
def role(symbol):  # 执行本行的状态、计算或校验逻辑。
    return "test" if symbol["path"].startswith("tests/") else "implementation"  # 执行本行的状态、计算或校验逻辑。
assembled = [symbol_by_id[item_id] for item_id in expanded_ids] + [symbol_by_id["s3"]]  # 执行本行的状态、计算或校验逻辑。
assert [role(symbol) for symbol in assembled] == ["implementation", "implementation", "test"]  # 执行本行的状态、计算或校验逻辑。
assert assembled[-1]["name"] == "test_limit"  # 执行本行的状态、计算或校验逻辑。
assert len(assembled) == 3  # 执行本行的状态、计算或校验逻辑。

## 6. revision gate 阻止过期片段进入 patch 规划

生成时的 HEAD、索引 revision 和工作树版本必须可比。文件变更后，旧 chunk 即使语义最相似也不能作为可修改代码；应增量重索引或要求重新读取目标文件，再生成 diff。


In [ ]:
def fresh(symbol, head_revision):  # 执行本行的状态、计算或校验逻辑。
    return symbol["revision"] == head_revision  # 执行本行的状态、计算或校验逻辑。
assert all(fresh(symbol, "c1") for symbol in assembled)  # 执行本行的状态、计算或校验逻辑。
assert not fresh(symbol_by_id["s1"], "c2")  # 执行本行的状态、计算或校验逻辑。
assert not fresh({**symbol_by_id["s2"], "revision": "old"}, "c1")  # 执行本行的状态、计算或校验逻辑。

## 7. patch/回答引用 path、行范围与 commit

Code RAG 最终输出必须能定位到实际符号，而非只输出“参考 refund 模块”。对修复任务还应保存 base commit、diff、测试命令和结果；当代码无法执行时，应明确只是静态建议。


In [ ]:
def citation(symbol):  # 执行本行的状态、计算或校验逻辑。
    return {"path": symbol["path"], "symbol": symbol["name"], "range": symbol["range"], "revision": symbol["revision"]}  # 执行本行的状态、计算或校验逻辑。
cite = citation(symbol_by_id["s2"] )  # 执行本行的状态、计算或校验逻辑。
assert cite["path"] == "limits.py"  # 执行本行的状态、计算或校验逻辑。
assert cite["range"] == (3, 8)  # 执行本行的状态、计算或校验逻辑。
assert cite["revision"] == "c1"  # 执行本行的状态、计算或校验逻辑。

## 8. 评测按任务与上下文阶段拆开

应分别测 direct symbol recall、dependency coverage、stale-context rate、测试发现率、patch apply/test pass 和 token/延迟。检索更多文件可能降低生成质量，选择性检索和不检索基线都应保留。


In [ ]:
def coverage(required, observed):  # 执行本行的状态、计算或校验逻辑。
    return set(required).issubset(set(observed))  # 执行本行的状态、计算或校验逻辑。
assert coverage(["s1", "s2"], expanded_ids)  # 执行本行的状态、计算或校验逻辑。
assert not coverage(["s1", "s3"], expanded_ids)  # 执行本行的状态、计算或校验逻辑。
assert all(citation(symbol)["revision"] == "c1" for symbol in assembled)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试应先说代码 RAG 的最小单位是 symbol+revision，再说明 direct retrieval、依赖展开、测试角色、token budget、工作树 freshness 和可执行验证。核心风险不是“没找到相似文本”，而是把过期或无关 API 当成当前契约。
